In [ ]:
import argparse
import csv
import hashlib
import json
import logging
import re
import os
import time
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup


# SETTINGS
SEARCH_BASE_URL = (
    "https://wuzzuf.net/saudi/search/jobs"
    "?filters%5Bcountry%5D%5B0%5D=Saudi%20Arabia"
)

SEARCH_PARAMS = {
    "q": "",
    "a": "spbg"
}

JSON_OUTPUT = "../data/raw/wuzzuf_jobs.json"
CSV_OUTPUT = "../data/raw/wuzzuf_jobs.csv"

REQUEST_TIMEOUT = 20
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = 5

LISTING_PAGE_DELAY = 3.0
DETAIL_PAGE_DELAY = 0.5

DEFAULT_WORKERS = 4


# HTTP HEADERS
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9,ar;q=0.5",
    "Accept": (
        "text/html,application/xhtml+xml,"
        "application/xml;q=0.9,*/*;q=0.8"
    ),
}


# WUZZUF STORE STATE PATTERNS
STORE_STATE_PATTERN = re.compile(
    r"Wuzzuf\.initialStoreState\s*=\s*(\{.*?\})\s*;\s*\n",
    re.DOTALL,
)

STORE_STATE_FALLBACK_PATTERN = re.compile(
    r"Wuzzuf\.initialStoreState\s*=\s*(\{.*\})",
    re.DOTALL,
)


# LOGGING
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

log = logging.getLogger("wuzzuf")


# CREATE REQUEST SESSION
def make_session():
    session = requests.Session()
    session.headers.update(HEADERS)
    return session


# FETCH HTML WITH RETRIES
def fetch_html(session, url, params=None):
    """
    GET a URL with retries and backoff.

    Returns:
        str: HTML content
        None: if all attempts fail
    """

    for attempt in range(1, MAX_RETRIES + 1):

        try:

            response = session.get(
                url,
                params=params,
                timeout=REQUEST_TIMEOUT
            )

            if response.status_code != 200:

                log.warning(
                    "Non-200 (%s) for %s "
                    "(attempt %d/%d)",
                    response.status_code,
                    response.url,
                    attempt,
                    MAX_RETRIES
                )

                # Retry rate-limit / forbidden responses
                if response.status_code in (403, 429):

                    time.sleep(
                        RETRY_BACKOFF_SECONDS * attempt
                    )

                    continue

                return None

            response.encoding = (
                response.apparent_encoding
                or "utf-8"
            )

            return response.text

        except requests.exceptions.RequestException as e:

            log.warning(
                "Request failed (%s) "
                "attempt %d/%d: %s",
                url,
                attempt,
                MAX_RETRIES,
                e
            )

            time.sleep(
                RETRY_BACKOFF_SECONDS * attempt
            )

    return None


# STAGE 1
# COLLECT JOB URLS FROM SEARCH RESULTS

def collect_job_urls(
    session,
    max_pages=None
):

    """
    Paginate through Wuzzuf search results
    and collect unique job-detail URLs.

    Stops when:
    - max_pages is reached
    - no new job URLs are found
    - request fails
    """

    job_urls = []
    seen = set()

    page = 0

    while True:

        # CHECK MAX PAGES

        if (
            max_pages is not None
            and page >= max_pages
        ):

            log.info(
                "Reached max_pages=%d. "
                "Stopping URL collection.",
                max_pages
            )

            break

        # SEARCH PARAMETERS
        params = dict(SEARCH_PARAMS)

        if page > 0:
            params["start"] = page

        log.info(
            "Scraping search page %d...",
            page + 1
        )

     
        # DOWNLOAD SEARCH PAGE
        html = fetch_html(
            session,
            SEARCH_BASE_URL,
            params=params
        )

        if html is None:

            log.error(
                "Search page %d failed after retries. "
                "Stopping.",
                page + 1
            )

            break

     
        # PARSE LINKS
     

        soup = BeautifulSoup(
            html,
            "html.parser"
        )

        page_urls = []

        for a in soup.find_all(
            "a",
            href=True
        ):

            href = a["href"]

            if "/jobs/p/" in href:

                full_url = urljoin(
                    "https://wuzzuf.net",
                    href
                )

                page_urls.append(
                    full_url
                )

     
        # REMOVE DUPLICATES
     

        new_urls = [
            url
            for url in dict.fromkeys(page_urls)
            if url not in seen
        ]

        log.info(
            "  found %d links (%d new)",
            len(page_urls),
            len(new_urls)
        )

     
        # STOP IF NO NEW JOBS
     

        if not new_urls:

            log.info(
                "No new jobs on this page. "
                "Stopping pagination."
            )

            break

     
        # ADD NEW URLS
     

        for url in new_urls:

            seen.add(url)
            job_urls.append(url)

        log.info(
            "  running total: %d unique job URLs",
            len(job_urls)
        )

        page += 1

     
        # POLITE DELAY
     

        time.sleep(
            LISTING_PAGE_DELAY
        )

    return job_urls


   
# STAGE 2
# EXTRACT WUZZUF STORE STATE
   

def extract_store_state(html):

    """
    Wuzzuf embeds job data inside:
        Wuzzuf.initialStoreState

    Extract and parse that JSON.
    """

    match = STORE_STATE_PATTERN.search(
        html
    )

    if not match:

        match = STORE_STATE_FALLBACK_PATTERN.search(
            html
        )

    if not match:

        raise ValueError(
            "Could not find "
            "'Wuzzuf.initialStoreState' "
            "in the page HTML."
        )

    raw_json = match.group(1)

    try:

        return json.loads(
            raw_json
        )

    except json.JSONDecodeError:

        return json.loads(
            _trim_to_balanced_json(
                raw_json
            )
        )


   
# TRIM JSON TO BALANCED OBJECT
   

def _trim_to_balanced_json(text):

    start = text.find("{")

    if start == -1:

        raise ValueError(
            "No opening brace found "
            "in store-state blob."
        )

    depth = 0
    in_string = False
    escape = False

    for i in range(
        start,
        len(text)
    ):

        char = text[i]

        if in_string:

            if escape:

                escape = False

            elif char == "\\":

                escape = True

            elif char == '"':

                in_string = False

            continue

        if char == '"':

            in_string = True

        elif char == "{":

            depth += 1

        elif char == "}":

            depth -= 1

            if depth == 0:

                return text[
                    start:i + 1
                ]

    raise ValueError(
        "Could not find a balanced "
        "closing brace for the JSON blob."
    )


   
# LOCALIZATION HELPER
   

def localized(
    node,
    field,
    fallback_field=None
):

    """
    Extract English translation from Wuzzuf
    localized objects.
    """

    if not isinstance(
        node,
        dict
    ):

        return None

    translations = (
        node.get("translations")
        or {}
    )

    field_translations = (
        translations.get(field)
    )

    if (
        isinstance(
            field_translations,
            dict
        )
        and field_translations.get("en")
    ):

        return field_translations["en"]

    if node.get(field):

        return node[field]

    if (
        fallback_field
        and node.get(fallback_field)
    ):

        return node[
            fallback_field
        ]

    return None


   
# HTML TO TEXT
   

def html_to_bullets(
    html_text
):

    """
    Convert HTML job description
    into readable plain text.
    """

    if not html_text:

        return []

    soup = BeautifulSoup(
        html_text,
        "html.parser"
    )

    # If the description contains <li>,
    # extract each bullet.

    items = soup.find_all("li")

    if items:

        bullets = [
            li.get_text(
                " ",
                strip=True
            )
            for li in items
            if li.get_text(
                strip=True
            )
        ]

        if bullets:

            return bullets

    # Otherwise extract normal text
    text = soup.get_text(
        "\n",
        strip=True
    )

    return [
        line
        for line in text.split("\n")
        if line
    ]


   
# FORMAT EXPERIENCE
   

def format_experience(
    years_obj
):

    """
    Convert Wuzzuf experience object
    into standardized string.

    Examples:
        1-2 years
        3+ years
        2 years
    """

    if not isinstance(
        years_obj,
        dict
    ):

        return None

    lo = years_obj.get("min")
    hi = years_obj.get("max")

    if (
        lo is None
        and hi is None
    ):

        return None

    if lo is None:

        return f"0 - {hi} years"

    if hi is None:

        return f"{lo}+ years"

    if lo == hi:

        return f"{lo} years"

    return f"{lo}-{hi} years"


   
# FORMAT SALARY
   

def format_salary(
    salary_obj
):

    """
    Convert Wuzzuf salary object
    into a standardized String.
    """

    if not isinstance(
        salary_obj,
        dict
    ):

        return None

    # UNPAID
    if not salary_obj.get(
        "isPaid",
        True
    ):

        return "Unpaid"

    # SALARY RANGE

    lo = salary_obj.get(
        "min"
    )

    hi = salary_obj.get(
        "max"
    )

    # CURRENCY

    currency = (
        salary_obj.get(
            "currency"
        )
        or {}
    )

    currency_code = (
        currency.get("code")
        or localized(
            currency,
            "name"
        )
    )

    # PERIOD

    period = (
        salary_obj.get(
            "period"
        )
        or {}
    )

    period_text = (
        localized(
            period,
            "period"
        )
        or period.get("name")
    )

    # ADDITIONAL DETAILS

    extra = salary_obj.get(
        "additionalDetails"
    )

    # BUILD BASE SALARY

    if (
        lo is None
        and hi is None
    ):

        base = "Confidential"

    elif (
        lo is not None
        and hi is not None
    ):

        base = (
            f"{lo:,}-{hi:,}"
        )

    else:

        base = f"{lo or hi:,}"

    # ADD CURRENCY

    parts = [base]

    if (
        currency_code
        and base != "Confidential"
    ):

        parts.append(
            currency_code
        )

    # ADD PERIOD

    if (
        period_text
        and base != "Confidential"
    ):

        parts.append(
            period_text
        )

    text = " ".join(
        parts
    )

    # ADD EXTRA DETAILS
    if extra:

        text = (
            f"{text} ({extra})"
        )

    return text


# FORMAT TIMESTAMP

def format_timestamp(
    value
):

    """
    Convert Wuzzuf timestamp into:

        YYYY-MM-DD HH:MM:SS

    Example:

        2026-10-25 15:30:00
    """

    if not value:

        return None

    try:

        value = str(value)

        # Convert Z to UTC offset
        value = value.replace(
            "Z",
            "+00:00"
        )

        dt = datetime.fromisoformat(
            value
        )

        # Convert timezone-aware
        # timestamp to UTC
        if dt.tzinfo is not None:

            dt = dt.astimezone(
                timezone.utc
            ).replace(
                tzinfo=None
            )

        return dt.strftime(
            "%Y-%m-%d %H:%M:%S"
        )

    except (
        ValueError,
        TypeError
    ):

        return str(value)


# GENERATE JOB ID

def generate_job_id(
    job_url
):

    """
    Generate a stable String job_id
    from the Wuzzuf job URL.

    Example:
        1021abc12345
    """

    slug_match = re.search(
        r"/jobs/p/([^/?]+)",
        job_url
    )

    if slug_match:

        slug = slug_match.group(1)

        return hashlib.sha1(
            slug.encode("utf-8")
        ).hexdigest()[:12]

    return hashlib.sha1(
        job_url.encode("utf-8")
    ).hexdigest()[:12]


# FIND JOB ENTITY
   

def find_job_id_by_slug(
    store,
    url
):

    """
    Match the correct job entity
    using the URL slug.
    """

    slug_match = re.search(
        r"/p/([^/?]+)",
        url
    )

    target_slug = (
        slug_match.group(1)
        if slug_match
        else None
    )

    job_collection = (
        store
        .get("entities", {})
        .get("job", {})
        .get("collection", {})
    )

    # Find exact job using slug

    if target_slug:

        for (
            job_id,
            job_entity
        ) in job_collection.items():

            attrs = job_entity.get(
                "attributes",
                {}
            )

            if (
                attrs.get("slug")
                == target_slug
            ):

                return (
                    job_id,
                    job_entity
                )

  
    # Fallback to similar jobs

    similar_jobs = (
        store
        .get("jobPage", {})
        .get("similarJobs", {})
    )

    if similar_jobs:

        main_job_id = next(
            iter(similar_jobs)
        )

        job_entity = (
            job_collection.get(
                main_job_id
            )
        )

        if job_entity:

            return (
                main_job_id,
                job_entity
            )

  
    # Final fallback
  

    if job_collection:

        job_id = next(
            iter(job_collection)
        )

        return (
            job_id,
            job_collection[job_id]
        )

    return None, None


   
# GET COMPANY
   

def get_company(
    store,
    job_entity
):

    company_ref = (
        job_entity
        .get("relationships", {})
        .get("company", {})
        .get("data", {})
    )

    if not company_ref:

        return None

    company_id = (
        company_ref.get("id")
    )

    return (
        store
        .get("entities", {})
        .get("company", {})
        .get("collection", {})
        .get(company_id)
    )


   
# NORMALIZE JOB
   

def normalize_job(
    job_entity,
    company_entity,
    job_url
):

    """
    Convert Wuzzuf raw data into
    the standardized final dataset.

    FINAL SCHEMA:

        job_id
        source
        job_title
        company_name
        job_description
        job_url
        country
        city
        work_mode
        employment_type
        career_level
        experience_needed
        qualification_required
        skills
        salary
        estimated_salary
        posted_date
        collected_at
    """

    attrs = job_entity.get(
        "attributes",
        {}
    )

    user_translations = attrs.get(
        "userContentTranslations",
        {}
    )

    
    # JOB TITLE
    

    title = (
        (
            user_translations
            .get("title")
            or {}
        )
        .get("en")
        or attrs.get("title")
    )

    
    # COMPANY NAME
    

    company_attrs = (
        (company_entity or {})
        .get("attributes", {})
    )

    company_name = (
        company_attrs.get(
            "name"
        )
    )

    
    # LOCATION
    

    location = (
        attrs.get(
            "location"
        )
        or {}
    )

    city_obj = (
        location.get(
            "city"
        )
        or {}
    )

    country_obj = (
        location.get(
            "country"
        )
        or {}
    )

    city = localized(
        city_obj,
        "name"
    )

    country = localized(
        country_obj,
        "name"
    )

    
    # WORK MODE
    

    workplace = (
        attrs.get(
            "workplaceArrangement"
        )
        or {}
    )

    work_mode = localized(
        workplace,
        "displayedName"
    )

    
    # EMPLOYMENT TYPE
    

    work_types = (
        attrs.get(
            "workTypes"
        )
        or []
    )

    employment_type = ", ".join(
        filter(
            None,
            (
                localized(
                    wt,
                    "displayedName"
                )
                for wt in work_types
            )
        )
    ) or None

    
    # CAREER LEVEL
    

    career_level = (
        attrs.get(
            "careerLevel"
        )
        or {}
    )

    career_level_text = (
        localized(
            career_level,
            "displayName"
        )
        or localized(
            career_level,
            "name"
        )
    )

    
    # QUALIFICATION
    

    education_level = (
        attrs
        .get(
            "candidatePreferences",
            {}
        )
        .get(
            "educationLevel"
        )
        or {}
    )

    qualification = localized(
        education_level,
        "name"
    )

    
    # EXPERIENCE
    

    experience_needed = (
        format_experience(
            attrs.get(
                "workExperienceYears"
            )
        )
    )

    
    # SKILLS
    

    keywords = (
        attrs.get(
            "keywords"
        )
        or []
    )

    skills = [
        k.get("name")
        for k in keywords
        if k.get("name")
    ]

    
    # SALARY
    

    salary = format_salary(
        attrs.get(
            "salary"
        )
    )

    
    # ESTIMATED SALARY
    
    #
    # Wuzzuf does not provide an estimated salary
    # that we should treat as the actual source value.
    #
    # Keep this NULL for now.
    #
    # Later, your ETL/ML process can populate this field.
    

    estimated_salary = None

    
    # JOB DESCRIPTION
    

    description_list = html_to_bullets(
        attrs.get(
            "description"
        )
    )

    job_description = (
        "\n".join(
            description_list
        )
        if description_list
        else None
    )

    
    # POSTED DATE
    

    posted_date = format_timestamp(
        attrs.get(
            "postedAt"
        )
    )

    
    # COLLECTED AT
    

    collected_at = (
        datetime.now(
            timezone.utc
        )
        .replace(
            tzinfo=None
        )
        .strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    )

    
    # FINAL STANDARDIZED RECORD
    

    return {

        "job_id": generate_job_id(
            job_url
        ),

        "source": "Wuzzuf",

        "job_title": title,

        "company_name": company_name,

        "job_description": job_description,

        "job_url": job_url,

        "country": country,

        "city": city,

        "work_mode": work_mode,

        "employment_type": employment_type,

        "career_level": career_level_text,

        "experience_needed": experience_needed,

        "qualification_required": qualification,

        "skills": skills,

        "salary": salary,

        "estimated_salary": estimated_salary,

        "posted_date": posted_date,

        "collected_at": collected_at,
    }


   
# SCRAPE ONE JOB DETAIL PAGE
   

def scrape_job_detail(
    session,
    url
):

    """
    Fetch and parse one job page.

    Returns:
        standardized job dictionary
        or None if scraping fails
    """

    try:

     
        # DOWNLOAD
     

        html = fetch_html(
            session,
            url
        )

        if html is None:

            log.error(
                "Failed to download job page: %s",
                url
            )

            return None

     
        # EXTRACT STORE STATE
     

        store_state = extract_store_state(
            html
        )

     
        # FIND JOB
     

        wuzzuf_job_id, job_entity = (
            find_job_id_by_slug(
                store_state,
                url
            )
        )

        if not job_entity:

            log.error(
                "Could not locate job entity "
                "in store state: %s",
                url
            )

            return None

     
        # FIND COMPANY
     

        company_entity = get_company(
            store_state,
            job_entity
        )

     
        # NORMALIZE
     

        return normalize_job(
            job_entity,
            company_entity,
            url
        )

    except Exception as e:

        log.error(
            "Error parsing %s: %s: %s",
            url,
            type(e).__name__,
            e
        )

        return None


   
# SAVE JSON
   

def save_json(
    records,
    filename
):

    dirname = os.path.dirname(
        filename
    )

    if dirname:

        os.makedirs(
            dirname,
            exist_ok=True
        )

    with open(
        filename,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            records,
            f,
            ensure_ascii=False,
            indent=2
        )


   
# CONVERT VALUE FOR CSV
   

def _stringify_csv_value(
    value
):

    if value is None:

        return ""

  
    # Skills:
    #
    # JSON:
    # ["Python", "SQL", "Snowflake"]
    #
    # CSV:
    # Python; SQL; Snowflake
  

    if isinstance(
        value,
        list
    ):

        return "; ".join(
            str(item)
            for item in value
        )

  
    # Dictionaries
  

    if isinstance(
        value,
        dict
    ):

        return json.dumps(
            value,
            ensure_ascii=False
        )

    return value


   
# SAVE CSV
   

def save_csv(
    records,
    filename
):

    if not records:

        return

    dirname = os.path.dirname(
        filename
    )

    if dirname:

        os.makedirs(
            dirname,
            exist_ok=True
        )

    
    # EXACT FINAL DATASET COLUMN ORDER
    

    fieldnames = [

        "job_id",

        "source",

        "job_title",

        "company_name",

        "job_description",

        "job_url",

        "country",

        "city",

        "work_mode",

        "employment_type",

        "career_level",

        "experience_needed",

        "qualification_required",

        "skills",

        "salary",

        "estimated_salary",

        "posted_date",

        "collected_at",
    ]

    
    # WRITE CSV
    

    with open(
        filename,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
            extrasaction="ignore"
        )

        writer.writeheader()

        for record in records:

            writer.writerow(
                {
                    column: _stringify_csv_value(
                        record.get(column)
                    )
                    for column in fieldnames
                }
            )


   
# SAVE FAILED URLS
   

def save_urls(
    urls,
    filename
):

    dirname = os.path.dirname(
        filename
    )

    if dirname:

        os.makedirs(
            dirname,
            exist_ok=True
        )

    with open(
        filename,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            "\n".join(urls)
        )


   
# MAIN SCRAPER
   

def run_scraper(
    max_pages=None,
    max_jobs=None,
    workers=DEFAULT_WORKERS
):

    """
    Main Wuzzuf scraping pipeline.

    Stage 1:
        Collect job URLs.

    Stage 2:
        Scrape job details.

    Stage 3:
        Save JSON + CSV.
    """

    session = make_session()

    
    # STAGE 1
    

    log.info("=" * 70)

    log.info(
        "STAGE 1: Collecting job URLs "
        "from search results"
    )

    log.info("=" * 70)

    job_urls = collect_job_urls(
        session,
        max_pages=max_pages
    )

    
    # LIMIT JOBS
    

    if max_jobs is not None:

        job_urls = job_urls[
            :max_jobs
        ]

        log.info(
            "Limiting to first %d jobs "
            "for detail scraping.",
            len(job_urls)
        )

    
    # CHECK RESULTS
    

    if not job_urls:

        log.warning(
            "No job URLs found. "
            "Nothing to scrape."
        )

        return [], []

    
    # STAGE 2
    

    log.info("=" * 70)

    log.info(
        "STAGE 2: Scraping details for %d jobs "
        "(workers=%d)",
        len(job_urls),
        workers
    )

    log.info("=" * 70)

    records = []

    failed_urls = []

    
    # WORKER
    

    def worker(url):

        # Each thread gets its own session
        thread_session = make_session()

        result = scrape_job_detail(
            thread_session,
            url
        )

        # Small delay between detail requests
        time.sleep(
            DETAIL_PAGE_DELAY
        )

        return (
            url,
            result
        )

    
    # THREAD POOL
    

    with ThreadPoolExecutor(
        max_workers=workers
    ) as executor:

        futures = {
            executor.submit(
                worker,
                url
            ): url

            for url in job_urls
        }

        done = 0

        for future in as_completed(
            futures
        ):

            url, result = (
                future.result()
            )

            done += 1

      
            # SUCCESS
      

            if result:

                records.append(
                    result
                )

                log.info(
                    "[%d/%d] OK - %s",
                    done,
                    len(job_urls),
                    result.get(
                        "job_title"
                    ) or url
                )

      
            # FAILURE
      

            else:

                failed_urls.append(
                    url
                )

                log.info(
                    "[%d/%d] FAIL - %s",
                    done,
                    len(job_urls),
                    url
                )

      
            # SAVE PROGRESS
      

            if (
                done % 10 == 0
                or done == len(job_urls)
            ):

                save_json(
                    records,
                    JSON_OUTPUT
                )

                save_csv(
                    records,
                    CSV_OUTPUT
                )

    
    # FINAL SAVE
    

    save_json(
        records,
        JSON_OUTPUT
    )

    save_csv(
        records,
        CSV_OUTPUT
    )

    
    # SUMMARY
    

    log.info("=" * 70)

    log.info(
        "DONE"
    )

    log.info("=" * 70)

    log.info(
        "Scraped: %d / %d jobs",
        len(records),
        len(job_urls)
    )

    log.info(
        "Failed: %d jobs",
        len(failed_urls)
    )

    log.info(
        "JSON out: %s",
        JSON_OUTPUT
    )

    log.info(
        "CSV out: %s",
        CSV_OUTPUT
    )

    
    # SAVE FAILED URLS
    

    if failed_urls:

        failed_file = (
            "wuzzuf_failed_urls.txt"
        )

        save_urls(
            failed_urls,
            failed_file
        )

        log.info(
            "Failed URLs saved to: %s",
            failed_file
        )

    return (
        records,
        failed_urls
    )


# COMMAND LINE ENTRY POINT

def main():

    parser = argparse.ArgumentParser(
        description=(
            "Scrape Wuzzuf job listings "
            "and job details."
        )
    )

    # MAX PAGES
    parser.add_argument(
        "--max-pages",
        type=int,
        default=None,
        help=(
            "Stop after this many "
            "search-result pages."
        )
    )

    # MAX JOBS
    parser.add_argument(
        "--max-jobs",
        type=int,
        default=None,
        help=(
            "Only scrape details for "
            "this many jobs."
        )
    )

    # WORKERS
    parser.add_argument(
        "--workers",
        type=int,
        default=DEFAULT_WORKERS,
        help=(
            f"Concurrent job-detail "
            f"fetch threads "
            f"(default: {DEFAULT_WORKERS})."
        )
    )

    # NOTEBOOK COMPATIBILITY
    args, _unknown = (
        parser.parse_known_args()
    )

    # RUN
    run_scraper(
        max_pages=args.max_pages,
        max_jobs=args.max_jobs,
        workers=args.workers
    )


# RUN SCRIPT

if __name__ == "__main__":
    main()



16:32:19 [INFO] ======================================================================
16:32:19 [INFO] STAGE 1: Collecting job URLs from search results
16:32:19 [INFO] ======================================================================
16:32:19 [INFO] Scraping search page 1...
16:32:44 [INFO]   found 15 links (15 new)
16:32:44 [INFO]   running total: 15 unique job URLs
16:32:47 [INFO] Scraping search page 2...
16:32:52 [INFO]   found 15 links (15 new)
16:32:52 [INFO]   running total: 30 unique job URLs
16:32:55 [INFO] Scraping search page 3...
16:32:58 [INFO]   found 15 links (15 new)
16:32:58 [INFO]   running total: 45 unique job URLs
16:33:01 [INFO] Scraping search page 4...
16:33:06 [INFO]   found 15 links (15 new)
16:33:06 [INFO]   running total: 60 unique job URLs
16:33:09 [INFO] Scraping search page 5...
16:33:14 [INFO]   found 15 links (15 new)
16:33:14 [INFO]   running total: 75 unique job URLs
16:33:17 [INFO] Scraping search page 6...
16:33:20 [INFO]   found 13 links (13 ne